# Preprocessing — UTA-RLDD (Clip Generation + Labelling)

Generates standardised clips for the UTA-RLDD dataset (the primary internal dataset) using ffmpeg, and builds labelled, class-capped train/val/test manifests for the subject-independent protocol. Clips are extracted at **10 fps, 112×112, 16 frames, stride 8** and saved as `.npz` arrays (Section 3.5).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
UTA_INPUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/UTA"   # <-- change if needed
UTA_OUT_DIR   = "/content/drive/MyDrive/ES327_Drowsiness/Clips/UTA_112_10fps_16f_stride8"

In [ ]:
!pip -q install opencv-python numpy pandas tqdm

In [ ]:
import os

SCRIPT_PATH = "/content/drive/MyDrive/ES327_Drowsiness/PythonScripts/preprocess_clips.py"
os.makedirs(os.path.dirname(SCRIPT_PATH), exist_ok=True)

script = r'''
import argparse
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".mpeg", ".mpg", ".webm"}

def list_videos(root: Path):
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS])

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def resample_indices(num_frames: int, src_fps: float, tgt_fps: float) -> np.ndarray:
    if num_frames <= 0:
        return np.array([], dtype=np.int32)
    if src_fps is None or src_fps <= 0:
        return np.arange(num_frames, dtype=np.int32)

    step = src_fps / tgt_fps
    idxs, t = [], 0.0
    while int(round(t)) < num_frames:
        idxs.append(int(round(t)))
        t += step

    idxs = np.array(idxs, dtype=np.int32)
    idxs = np.clip(idxs, 0, num_frames - 1)
    return np.unique(idxs)

def read_all_frames(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frames.append(frame_bgr)
    cap.release()
    return fps, frames

def preprocess_video_to_clips(video_path: Path, out_dir: Path, target_fps: float,
                              clip_len: int, stride: int, size: int, layout: str, video_id: str):
    fps, frames_bgr = read_all_frames(video_path)
    n = len(frames_bgr)
    if n == 0:
        return []

    idxs = resample_indices(n, fps, target_fps)
    if idxs.size < clip_len:
        return []

    resampled = []
    for i in idxs:
        frame = frames_bgr[int(i)]
        frame = cv2.resize(frame, (size, size), interpolation=cv2.INTER_AREA)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resampled.append(frame)
    resampled = np.stack(resampled, axis=0).astype(np.uint8)  # (T,H,W,C)

    T = resampled.shape[0]
    clip_paths = []
    clip_idx = 0
    for start in range(0, T - clip_len + 1, stride):
        clip = resampled[start:start + clip_len]  # (16,112,112,3)

        if layout.upper() == "TCHW":
            clip = np.transpose(clip, (3, 0, 1, 2))  # (C,T,H,W)
        elif layout.upper() == "THWC":
            pass
        else:
            raise ValueError("layout must be 'TCHW' or 'THWC'")

        out_path = out_dir / f"{video_id}__{clip_idx:05d}.npz"
        np.savez_compressed(out_path, clip=clip)
        clip_paths.append(str(out_path))
        clip_idx += 1

    return clip_paths

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input_dir", required=True)
    ap.add_argument("--output_dir", required=True)
    ap.add_argument("--dataset", default="UNKNOWN")
    ap.add_argument("--target_fps", type=float, default=10.0)
    ap.add_argument("--clip_len", type=int, default=16)
    ap.add_argument("--stride", type=int, default=8)
    ap.add_argument("--size", type=int, default=112)
    ap.add_argument("--layout", choices=["TCHW", "THWC"], default="TCHW")
    args = ap.parse_args()

    in_root = Path(args.input_dir).expanduser().resolve()
    out_root = Path(args.output_dir).expanduser().resolve()
    out_clips = out_root / "clips_npz"
    safe_mkdir(out_clips)

    videos = list_videos(in_root)
    if not videos:
        raise SystemExit(f"No videos found in: {in_root}")

    rows = []
    for vp in tqdm(videos, desc=f"Preprocessing {args.dataset}"):
        rel = vp.relative_to(in_root)
        video_id = "__".join(rel.with_suffix("").parts)

        try:
            clip_paths = preprocess_video_to_clips(
                vp, out_clips, args.target_fps, args.clip_len, args.stride, args.size, args.layout, video_id
            )
            rows.append({
                "dataset": args.dataset,
                "video_path": str(vp),
                "clips_written": len(clip_paths),
                "error": ""
            })
        except Exception as e:
            rows.append({"dataset": args.dataset, "video_path": str(vp), "clips_written": 0, "error": str(e)})

    df = pd.DataFrame(rows)
    out_root.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_root / "manifest_videos.csv", index=False)

    print("\nDone.")
    print("Videos scanned:", len(videos))
    print("Succeeded:", int((df["clips_written"] > 0).sum()))
    print("Failed:", int((df["clips_written"] == 0).sum()))
    print("Total clips written:", int(df["clips_written"].sum()))
    print("Output folder:", out_root)

if __name__ == "__main__":
    main()
'''
with open(SCRIPT_PATH, "w") as f:
    f.write(script)

print("✅ Saved:", SCRIPT_PATH)

In [ ]:
!python "$SCRIPT_PATH" \
  --input_dir "$UTA_INPUT_DIR" \
  --output_dir "$UTA_OUT_DIR" \
  --dataset "UTA" \
  --target_fps 10 \
  --clip_len 16 \
  --stride 8 \
  --size 112 \
  --layout TCHW

In [ ]:
import glob, cv2

UTA_INPUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/UTA"
vids = sorted(glob.glob(UTA_INPUT_DIR + "/**/*.mov", recursive=True))
print("Found mov:", len(vids))
print("Example:", vids[0] if vids else None)

p = vids[0]
cap = cv2.VideoCapture(p)
print("Opened:", cap.isOpened())
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
cap.release()

In [ ]:
import glob
UTA_INPUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/UTA"
vids = sorted(glob.glob(UTA_INPUT_DIR + "/**/*.mov", recursive=True))
print("UTA MOVs currently on Drive:", len(vids))
print("First 10:")
for v in vids[:10]:
    print(v)

In [ ]:
import os

SCRIPT_PATH2 = "/content/drive/MyDrive/ES327_Drowsiness/PythonScripts/preprocess_uta_stream_resume.py"
os.makedirs(os.path.dirname(SCRIPT_PATH2), exist_ok=True)

script2 = r'''
import argparse
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".mpeg", ".mpg", ".webm"}

def list_videos(root: Path):
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS])

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def already_done(video_id: str, out_clips: Path):
    # if at least one clip exists, assume video already processed
    return any(out_clips.glob(f"{video_id}__*.npz"))

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input_dir", required=True)
    ap.add_argument("--output_dir", required=True)
    ap.add_argument("--dataset", default="UNKNOWN")
    ap.add_argument("--target_fps", type=float, default=10.0)
    ap.add_argument("--clip_len", type=int, default=16)
    ap.add_argument("--stride", type=int, default=8)
    ap.add_argument("--size", type=int, default=112)
    ap.add_argument("--layout", choices=["TCHW", "THWC"], default="TCHW")
    ap.add_argument("--max_frames", type=int, default=200000)
    ap.add_argument("--resume", action="store_true")
    args = ap.parse_args()

    in_root = Path(args.input_dir).expanduser().resolve()
    out_root = Path(args.output_dir).expanduser().resolve()
    out_clips = out_root / "clips_npz"
    safe_mkdir(out_clips)

    videos = list_videos(in_root)
    if not videos:
        raise SystemExit(f"No videos found in: {in_root}")

    # load existing video manifest to append to
    manifest_path = out_root / "manifest_videos.csv"
    existing = None
    if manifest_path.exists():
        existing = pd.read_csv(manifest_path)

    rows = []
    for vp in tqdm(videos, desc=f"Preprocessing {args.dataset}"):
        rel = vp.relative_to(in_root)
        video_id = "__".join(rel.with_suffix("").parts)

        if args.resume and already_done(video_id, out_clips):
            continue

        cap = cv2.VideoCapture(str(vp))
        if not cap.isOpened():
            rows.append({"dataset": args.dataset, "video_path": str(vp), "clips_written": 0, "error": "Could not open"})
            continue

        src_fps = cap.get(cv2.CAP_PROP_FPS)
        if src_fps is None or src_fps <= 0:
            src_fps = args.target_fps

        step = max(src_fps / args.target_fps, 1e-6)

        buffer = []
        clips_written = 0
        frame_i = 0
        next_t = 0.0
        safety = 0

        try:
            while True:
                ok, frame_bgr = cap.read()
                if not ok:
                    break

                if frame_i >= int(round(next_t)):
                    frame = cv2.resize(frame_bgr, (args.size, args.size), interpolation=cv2.INTER_AREA)
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    buffer.append(frame)
                    next_t += step

                    if len(buffer) >= args.clip_len:
                        if (len(buffer) - args.clip_len) % args.stride == 0:
                            clip = np.stack(buffer[-args.clip_len:], axis=0).astype(np.uint8)  # (T,H,W,C)
                            if args.layout.upper() == "TCHW":
                                clip = np.transpose(clip, (3,0,1,2))
                            out_path = out_clips / f"{video_id}__{clips_written:05d}.npz"
                            np.savez_compressed(out_path, clip=clip)
                            clips_written += 1

                frame_i += 1
                safety += 1
                if safety >= args.max_frames:
                    break

            cap.release()
            rows.append({"dataset": args.dataset, "video_path": str(vp), "clips_written": clips_written, "error": ""})

        except Exception as e:
            cap.release()
            rows.append({"dataset": args.dataset, "video_path": str(vp), "clips_written": 0, "error": str(e)})

    df_new = pd.DataFrame(rows)

    if existing is not None and len(df_new) > 0:
        df = pd.concat([existing, df_new], ignore_index=True)
        # keep last occurrence per video_path
        df = df.drop_duplicates(subset=["video_path"], keep="last")
    elif existing is not None:
        df = existing
    else:
        df = df_new

    out_root.mkdir(parents=True, exist_ok=True)
    df.to_csv(manifest_path, index=False)

    print("\nDone.")
    print("Videos scanned:", len(videos))
    print("Succeeded:", int((df["clips_written"] > 0).sum()))
    print("Failed:", int((df["clips_written"] == 0).sum()))
    print("Total clips written:", int(df["clips_written"].sum()))
    print("Output folder:", out_root)

if __name__ == "__main__":
    main()
'''
with open(SCRIPT_PATH2, "w") as f:
    f.write(script2)

print("✅ Saved:", SCRIPT_PATH2)

In [ ]:
UTA_OUT_DIR   = "/content/drive/MyDrive/ES327_Drowsiness/Clips/UTA_112_10fps_16f_stride8"

!python "$SCRIPT_PATH2" \
  --input_dir "$UTA_INPUT_DIR" \
  --output_dir "$UTA_OUT_DIR" \
  --dataset "UTA" \
  --target_fps 10 \
  --clip_len 16 \
  --stride 8 \
  --size 112 \
  --layout TCHW \
  --resume

In [ ]:
import os, glob
from collections import Counter

UTA_INPUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/UTA"

all_files = glob.glob(UTA_INPUT_DIR + "/**/*", recursive=True)
all_files = [f for f in all_files if os.path.isfile(f)]

exts = Counter([os.path.splitext(f)[1].lower() for f in all_files])
print("Extension counts:")
print(exts)

video_exts = {".mov",".mp4",".avi",".mkv",".mpeg",".mpg",".webm"}
videos = [f for f in all_files if os.path.splitext(f)[1].lower() in video_exts]

print("\nTotal video files found:", len(videos))
print("\nSample video paths:")
for v in sorted(videos)[:20]:
    print(v)

In [ ]:
import glob, os
from collections import Counter

UTA="/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/UTA"

# list all files
files = [p for p in glob.glob(UTA + "/**/*", recursive=True) if os.path.isfile(p)]
print("Total files:", len(files))

exts = Counter(os.path.splitext(p)[1].lower() for p in files)
print("Top extensions:", exts.most_common(10))

video_exts = {".mov",".mp4",".avi",".mkv",".mpeg",".mpg",".webm"}
vids = [p for p in files if os.path.splitext(p)[1].lower() in video_exts]
print("Total video files (any ext):", len(vids))
print("First 10 video paths:")
for v in sorted(vids)[:10]:
    print(v)

In [ ]:
script = r'''
import argparse, os, re, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

VIDEO_EXTS = {".mov", ".mp4", ".avi", ".mkv", ".mpeg", ".mpg", ".webm"}

def list_videos(root: Path):
    vids = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
            vids.append(p)
    return sorted(vids)

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def parse_uta_ids(video_path: Path):
    """
    UTA files live like .../<subject>/<stage>.mov where stage in {0,5,10}
    Some subjects may have duplicates; we keep a recording_id that is unique per file.
    """
    name = video_path.stem  # "0" or "5" or "10" (sometimes other)
    stage = None
    if name in {"0","5","10"}:
        stage = int(name)
    # subject folder typically parent name is like "01", "34", etc.
    subject = video_path.parent.name
    # label rule (binary): 10 => 1 (drowsy), 0/5 => 0 (not-drowsy)
    label = None
    if stage is not None:
        label = 1 if stage >= 10 else 0
    return subject, stage, label

def ffmpeg_decode_rgb(video_path: Path, fps: int, size: int):
    """
    Decode video -> raw RGB frames at specified fps and size using ffmpeg.
    Returns numpy array shaped (T, H, W, 3) uint8.
    """
    vf = f"fps={fps},scale={size}:{size}"
    cmd = [
        "ffmpeg",
        "-v", "error",
        "-i", str(video_path),
        "-vf", vf,
        "-f", "rawvideo",
        "-pix_fmt", "rgb24",
        "pipe:1"
    ]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if p.returncode != 0:
        err = p.stderr.decode("utf-8", errors="ignore")
        raise RuntimeError(f"ffmpeg failed: {err[:400]}")

    raw = p.stdout
    frame_bytes = size * size * 3
    if len(raw) < frame_bytes:
        return np.empty((0, size, size, 3), dtype=np.uint8)

    n_frames = len(raw) // frame_bytes
    raw = raw[:n_frames * frame_bytes]
    arr = np.frombuffer(raw, dtype=np.uint8)
    frames = arr.reshape((n_frames, size, size, 3))
    return frames

def save_clips(frames_thwc: np.ndarray, out_dir: Path, video_id: str, clip_len: int, stride: int, layout: str):
    """
    frames_thwc: (T,H,W,C) uint8
    saves npz clips to out_dir and returns list of written paths
    """
    T = frames_thwc.shape[0]
    if T < clip_len:
        return []

    written = []
    clip_idx = 0
    for start in range(0, T - clip_len + 1, stride):
        clip = frames_thwc[start:start+clip_len]  # (L,H,W,C)
        if layout.upper() == "TCHW":
            clip = np.transpose(clip, (3, 0, 1, 2))  # (C,L,H,W)

        out_path = out_dir / f"{video_id}__{clip_idx:05d}.npz"
        np.savez_compressed(out_path, clip=clip)
        written.append(str(out_path))
        clip_idx += 1
    return written

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input_dir", required=True)
    ap.add_argument("--output_dir", required=True)
    ap.add_argument("--dataset", default="UTA")
    ap.add_argument("--target_fps", type=int, default=10)
    ap.add_argument("--clip_len", type=int, default=16)
    ap.add_argument("--stride", type=int, default=8)
    ap.add_argument("--size", type=int, default=112)
    ap.add_argument("--layout", choices=["TCHW","THWC"], default="TCHW")
    ap.add_argument("--resume", action="store_true")
    args = ap.parse_args()

    in_root = Path(args.input_dir).expanduser().resolve()
    out_root = Path(args.output_dir).expanduser().resolve()
    clips_dir = out_root / "clips_npz"
    safe_mkdir(clips_dir)

    manifest_path = out_root / "manifest_uta.csv"
    done_videos = set()
    if args.resume and manifest_path.exists():
        try:
            old = pd.read_csv(manifest_path)
            done_videos = set(old["video_path"].astype(str).tolist())
            print(f"Resume enabled: {len(done_videos)} videos already in manifest.")
        except Exception:
            pass

    videos = list_videos(in_root)
    if not videos:
        raise SystemExit(f"No videos found in: {in_root}")

    rows = []
    for vp in tqdm(videos, desc=f"Preprocessing {args.dataset} (ffmpeg)"):
        if str(vp) in done_videos:
            continue

        subject, stage, label = parse_uta_ids(vp)

        # unique id based on relative path (keeps duplicates safe)
        rel = vp.relative_to(in_root)
        video_id = "__".join(rel.with_suffix("").parts)

        try:
            frames = ffmpeg_decode_rgb(vp, fps=args.target_fps, size=args.size)
            clip_paths = save_clips(frames, clips_dir, video_id, args.clip_len, args.stride, args.layout)

            for cp in clip_paths:
                rows.append({
                    "dataset": args.dataset,
                    "video_path": str(vp),
                    "clip_path": cp,
                    "subject_id": subject,
                    "stage": stage,          # 0 / 5 / 10 (or None if weird name)
                    "label": label,          # binary: 0/5 => 0, 10 => 1
                    "target_fps": args.target_fps,
                    "clip_len": args.clip_len,
                    "stride": args.stride,
                    "size": args.size,
                    "layout": args.layout,
                })

            # if no clips wrote, still log it
            if len(clip_paths) == 0:
                rows.append({
                    "dataset": args.dataset,
                    "video_path": str(vp),
                    "clip_path": "",
                    "subject_id": subject,
                    "stage": stage,
                    "label": label,
                    "error": "too_few_frames_after_decode"
                })

        except Exception as e:
            rows.append({
                "dataset": args.dataset,
                "video_path": str(vp),
                "clip_path": "",
                "subject_id": subject,
                "stage": stage,
                "label": label,
                "error": str(e)[:400]
            })

    df_new = pd.DataFrame(rows)

    if manifest_path.exists() and args.resume:
        df_old = pd.read_csv(manifest_path)
        df_out = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_out = df_new

    out_root.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(manifest_path, index=False)

    print("\nDone.")
    print("Videos scanned:", len(videos))
    print("New rows added:", len(df_new))
    print("Total manifest rows:", len(df_out))
    print("Clips written total:", int((df_out.get("clip_path","") != "").sum()))
    print("Manifest:", str(manifest_path))
    print("Clips dir:", str(clips_dir))

if __name__ == "__main__":
    main()
'''
with open("preprocess_uta_ffmpeg.py", "w") as f:
    f.write(script)
print("✅ wrote preprocess_uta_ffmpeg.py")

In [ ]:
UTA_INPUT_DIR  = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/UTA"
UTA_OUTPUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/Clips/UTA_112_10fps_16f_stride8"

In [ ]:
!python preprocess_uta_ffmpeg.py \
  --input_dir "$UTA_INPUT_DIR" \
  --output_dir "$UTA_OUTPUT_DIR" \
  --dataset "UTA" \
  --target_fps 10 \
  --clip_len 16 \
  --stride 8 \
  --size 112 \
  --layout TCHW \
  --resume

In [ ]:
script = r'''
import argparse, os, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

VIDEO_EXTS = {".mov", ".mp4", ".avi", ".mkv", ".mpeg", ".mpg", ".webm"}

def list_videos(root: Path):
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS])

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def parse_uta_ids(video_path: Path):
    # stage is filename stem: 0 / 5 / 10
    stem = video_path.stem
    stage = int(stem) if stem.isdigit() else None
    subject = video_path.parent.name  # folder name like "01"
    label = None
    if stage in (0, 5, 10):
        label = 1 if stage == 10 else 0
    return subject, stage, label

def stream_clips_ffmpeg(video_path: Path, out_dir: Path, video_id: str,
                        target_fps: int, size: int, clip_len: int, stride: int,
                        layout: str):
    """
    Stream raw RGB frames from ffmpeg (already downsampled fps+scaled), build sliding clips, save to npz.
    Constant memory. Returns (clips_written, total_frames_decoded).
    """
    frame_bytes = size * size * 3
    vf = f"fps={target_fps},scale={size}:{size}"

    cmd = [
        "ffmpeg", "-v", "error",
        "-i", str(video_path),
        "-vf", vf,
        "-f", "rawvideo",
        "-pix_fmt", "rgb24",
        "pipe:1"
    ]

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    assert proc.stdout is not None

    # sliding buffer of last clip_len frames (THWC)
    buf = np.empty((clip_len, size, size, 3), dtype=np.uint8)
    filled = 0
    frame_idx = 0
    clips_written = 0

    # To implement stride, we save a clip every `stride` frames once buffer full.
    # Equivalent to windows starting at 0, stride, 2*stride, ... in the downsampled stream.
    while True:
        raw = proc.stdout.read(frame_bytes)
        if not raw or len(raw) < frame_bytes:
            break

        frame = np.frombuffer(raw, dtype=np.uint8).reshape((size, size, 3))

        if filled < clip_len:
            buf[filled] = frame
            filled += 1
        else:
            # shift left by 1 and append
            buf[:-1] = buf[1:]
            buf[-1] = frame

        # once buffer full, decide whether to write this window
        if filled == clip_len:
            start = frame_idx - (clip_len - 1)  # start index of current window
            if start % stride == 0:
                clip = buf.copy()  # (L,H,W,C)
                if layout.upper() == "TCHW":
                    clip = np.transpose(clip, (3, 0, 1, 2))  # (C,L,H,W)

                out_path = out_dir / f"{video_id}__{clips_written:05d}.npz"
                np.savez_compressed(out_path, clip=clip)
                clips_written += 1

        frame_idx += 1

    # finish + check ffmpeg error if any
    proc.stdout.close()
    stderr = proc.stderr.read().decode("utf-8", errors="ignore") if proc.stderr else ""
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"ffmpeg failed ({ret}): {stderr[:400]}")

    return clips_written, frame_idx

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input_dir", required=True)
    ap.add_argument("--output_dir", required=True)
    ap.add_argument("--dataset", default="UTA")
    ap.add_argument("--target_fps", type=int, default=10)
    ap.add_argument("--clip_len", type=int, default=16)
    ap.add_argument("--stride", type=int, default=8)
    ap.add_argument("--size", type=int, default=112)
    ap.add_argument("--layout", choices=["TCHW","THWC"], default="TCHW")
    ap.add_argument("--resume", action="store_true")
    args = ap.parse_args()

    in_root = Path(args.input_dir).expanduser().resolve()
    out_root = Path(args.output_dir).expanduser().resolve()
    clips_dir = out_root / "clips_npz"
    safe_mkdir(clips_dir)

    manifest_path = out_root / "manifest_uta.csv"
    done_videos = set()
    if args.resume and manifest_path.exists():
        old = pd.read_csv(manifest_path)
        done_videos = set(old["video_path"].astype(str).unique().tolist())
        print(f"Resume: found {len(done_videos)} videos already processed.")

    videos = list_videos(in_root)
    if not videos:
        raise SystemExit(f"No videos found in: {in_root}")

    rows = []

    for vp in tqdm(videos, desc=f"Preprocessing {args.dataset} (streaming ffmpeg)", mininterval=0.5):
        if str(vp) in done_videos:
            continue

        subject, stage, label = parse_uta_ids(vp)
        rel = vp.relative_to(in_root)
        video_id = "__".join(rel.with_suffix("").parts)

        print(f"\nNow processing: {rel}")  # <-- immediate heartbeat per file

        try:
            clips_written, frames_decoded = stream_clips_ffmpeg(
                vp, clips_dir, video_id,
                target_fps=args.target_fps, size=args.size,
                clip_len=args.clip_len, stride=args.stride,
                layout=args.layout
            )

            if clips_written == 0:
                rows.append({
                    "dataset": args.dataset, "video_path": str(vp), "clip_path": "",
                    "subject_id": subject, "stage": stage, "label": label,
                    "error": f"no_clips (frames_decoded={frames_decoded})"
                })
            else:
                # we don’t record every clip row here (would be huge); store per-video summary
                rows.append({
                    "dataset": args.dataset, "video_path": str(vp), "clip_path": "__MULTI__",
                    "subject_id": subject, "stage": stage, "label": label,
                    "clips_written": clips_written, "frames_decoded": frames_decoded,
                    "target_fps": args.target_fps, "clip_len": args.clip_len,
                    "stride": args.stride, "size": args.size, "layout": args.layout
                })

        except Exception as e:
            rows.append({
                "dataset": args.dataset, "video_path": str(vp), "clip_path": "",
                "subject_id": subject, "stage": stage, "label": label,
                "error": str(e)[:400]
            })

        # append to CSV progressively so you don’t lose progress if Colab resets
        df_new = pd.DataFrame(rows)
        if manifest_path.exists() and args.resume:
            df_old = pd.read_csv(manifest_path)
            df_out = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_out = df_new
        out_root.mkdir(parents=True, exist_ok=True)
        df_out.to_csv(manifest_path, index=False)

        rows = []  # reset batch buffer after writing

    print("\n✅ Done.")
    print("Videos scanned:", len(videos))
    print("Manifest:", str(manifest_path))
    print("Clips dir:", str(clips_dir))

if __name__ == "__main__":
    main()
'''
with open("preprocess_uta_stream.py", "w") as f:
    f.write(script)
print("✅ wrote preprocess_uta_stream.py")

In [ ]:
UTA_INPUT_DIR  = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/UTA"
UTA_OUTPUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/Clips/UTA_112_10fps_16f_stride8"

!python preprocess_uta_stream.py \
  --input_dir "$UTA_INPUT_DIR" \
  --output_dir "$UTA_OUTPUT_DIR" \
  --dataset "UTA" \
  --target_fps 10 \
  --clip_len 16 \
  --stride 8 \
  --size 112 \
  --layout TCHW \
  --resume

In [ ]:
import os

manifest = "/content/drive/MyDrive/ES327_Drowsiness/Clips/UTA_112_10fps_16f_stride8/manifest.csv"
print("Manifest exists:", os.path.exists(manifest))

In [ ]:
import os, pandas as pd

UTA_OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/UTA_112_10fps_16f_stride8"

for name in ["manifest_uta.csv", "manifest.csv", "manifest uta.csv", "manifest_uta"]:
    p = os.path.join(UTA_OUT, name)
    if os.path.exists(p):
        print("FOUND:", p)

# load the one you said exists
manifest_path = os.path.join(UTA_OUT, "manifest_uta.csv")
dfm = pd.read_csv(manifest_path)

print("rows:", len(dfm))
print(dfm.columns)
print(dfm.head(3))

In [ ]:
import pandas as pd
import os

UTA_OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/UTA_112_10fps_16f_stride8"
manifest_path = os.path.join(UTA_OUT, "manifest_uta.csv")

dfm = pd.read_csv(manifest_path)

print("Columns:")
print(dfm.columns.tolist())

print("\nHead:")
display(dfm.head())

In [ ]:
done = dfm[dfm["clips_written"] > 0]

print("Completed videos:", done["video_path"].nunique())
print("Total logged videos:", dfm["video_path"].nunique())

In [ ]:
import pandas as pd, os

UTA_OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/UTA_112_10fps_16f_stride8"
manifest_path = os.path.join(UTA_OUT, "manifest_uta.csv")

dfm = pd.read_csv(manifest_path)

done_paths = set(dfm.loc[dfm["clips_written"] > 0, "video_path"].astype(str))
print("Done videos (from manifest):", len(done_paths))
print("Total rows in manifest:", len(dfm))

In [ ]:
from pathlib import Path

UTA_IN = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/UTA"
in_root = Path(UTA_IN)

videos = sorted([p for p in in_root.rglob("*") if p.is_file() and p.suffix.lower() in {".mov", ".mp4", ".avi", ".mkv"}])
print("Total videos found in input:", len(videos))

todo = [p for p in videos if str(p) not in done_paths]
print("Remaining (not in manifest done):", len(todo))
print("Example todo:", todo[0] if todo else "None")

In [ ]:
import glob, os, time

UTA_OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/UTA_112_10fps_16f_stride8"
clips_dir = os.path.join(UTA_OUT, "clips_npz")
os.makedirs(clips_dir, exist_ok=True)

files = glob.glob(clips_dir + "/*.npz")
print("Current npz clips:", len(files))
if files:
    newest = max(files, key=os.path.getmtime)
    print("Newest file:", os.path.basename(newest))
    print("Seconds since newest:", round(time.time() - os.path.getmtime(newest), 1))

In [ ]:
import os, subprocess, numpy as np
from tqdm import tqdm
from pathlib import Path
import pandas as pd

UTA_IN = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/UTA"
in_root = Path(UTA_IN)

UTA_OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/UTA_112_10fps_16f_stride8"
clips_dir = os.path.join(UTA_OUT, "clips_npz")
os.makedirs(clips_dir, exist_ok=True)

manifest_path = os.path.join(UTA_OUT, "manifest_uta.csv")
dfm = pd.read_csv(manifest_path)

done_paths = set(dfm.loc[dfm["clips_written"] > 0, "video_path"].astype(str))

# build todo again (using os.walk result if you already have it, otherwise rebuild)
videos = []
for root, dirs, files in os.walk(UTA_IN):
    for f in files:
        if f.lower().endswith((".mov", ".mp4", ".avi", ".mkv")):
            videos.append(os.path.join(root, f))

todo = [p for p in videos if p not in done_paths]
print("✅ Remaining to process:", len(todo))

def video_id_from_path(vp: str, root: Path) -> str:
    rel = Path(vp).relative_to(root)
    return "__".join(rel.with_suffix("").parts)

def extract_clips_ffmpeg(video_path: str, out_dir: str, video_id: str,
                         target_fps=10, clip_len=16, stride=8, size=112, layout="TCHW"):
    frame_bytes = size * size * 3
    cmd = [
        "ffmpeg", "-nostdin", "-v", "error", "-i", video_path,
        "-vf", f"fps={target_fps},scale={size}:{size}",
        "-f", "rawvideo", "-pix_fmt", "rgb24", "pipe:1"
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, bufsize=10**7)

    frames = []
    clip_idx = 0
    written = 0

    try:
        while True:
            buf = proc.stdout.read(frame_bytes)
            if not buf or len(buf) < frame_bytes:
                break

            frame = np.frombuffer(buf, dtype=np.uint8).reshape(size, size, 3)
            frames.append(frame)

            while len(frames) >= clip_len:
                clip = np.stack(frames[:clip_len], axis=0)  # (T,H,W,C)
                if layout.upper() == "TCHW":
                    clip = np.transpose(clip, (3,0,1,2))     # (C,T,H,W)

                out_path = os.path.join(out_dir, f"{video_id}__{clip_idx:05d}.npz")
                np.savez_compressed(out_path, clip=clip)

                written += 1
                clip_idx += 1
                frames = frames[stride:]

        proc.wait(timeout=30)
        err = proc.stderr.read().decode("utf-8", errors="ignore").strip()
        if proc.returncode != 0 and err:
            return written, err
        return written, ""

    except Exception as e:
        try: proc.kill()
        except: pass
        return written, str(e)

# Append rows with the same columns as your manifest
cols = dfm.columns.tolist()
def append_row(row_dict):
    row = {c: row_dict.get(c, "") for c in cols}
    pd.DataFrame([row]).to_csv(manifest_path, mode="a", header=False, index=False)

for vp in tqdm(todo, desc="UTA resume (manifest-based)"):
    vid = video_id_from_path(vp, in_root)
    print("Now processing:", Path(vp).relative_to(in_root), flush=True)

    written, err = extract_clips_ffmpeg(
        vp, clips_dir, vid,
        target_fps=10, clip_len=16, stride=8, size=112, layout="TCHW"
    )

    row = {"video_path": vp, "clips_written": int(written)}
    if "video_id" in cols: row["video_id"] = vid
    if "dataset" in cols: row["dataset"] = "UTA"
    if "error" in cols: row["error"] = err

    append_row(row)

In [ ]:
import pandas as pd
import os, glob

UTA_OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/UTA_112_10fps_16f_stride8"
manifest_path = os.path.join(UTA_OUT, "manifest_uta.csv")

df = pd.read_csv(manifest_path)

print("Total manifest rows:", len(df))
print("Completed videos:", (df["clips_written"] > 0).sum())

clips_dir = os.path.join(UTA_OUT, "clips_npz")
print("Total .npz clips:", len(glob.glob(clips_dir + "/*.npz")))

In [ ]:
import glob, os, pandas as pd

clips_dir = os.path.join(UTA_OUT, "clips_npz")
clips = glob.glob(clips_dir + "/*.npz")

rows = []

for fp in clips:
    base = os.path.basename(fp)
    video_id = base.rsplit("__", 1)[0]

    # video_id format: FoldX__subject__stage
    parts = video_id.split("__")
    stage = int(parts[-1])   # 0, 5, or 10
    subject = parts[-2]

    label = 1 if stage == 10 else 0

    rows.append({
        "clip_path": fp,
        "video_id": video_id,
        "subject_id": subject,
        "stage": stage,
        "label": label
    })

df_clips = pd.DataFrame(rows)

print("Total clips:", len(df_clips))
print("Label distribution:")
print(df_clips["label"].value_counts())

In [ ]:
import numpy as np

MAX_CLIPS = 400
SEED = 42

np.random.seed(SEED)

df_capped = (
    df_clips
    .groupby("video_id", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), MAX_CLIPS),
        random_state=SEED
    ))
)

print("After cap:", len(df_capped))
print(df_capped["label"].value_counts())

In [ ]:
counts = df_capped["label"].value_counts()
min_class = counts.min()

df_balanced = (
    df_capped
    .groupby("label", group_keys=False)
    .apply(lambda x: x.sample(n=min_class, random_state=SEED))
)

df_balanced = df_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("After balancing:", len(df_balanced))
print(df_balanced["label"].value_counts())

In [ ]:
train_manifest_path = os.path.join(UTA_OUT, "manifest_uta_train_capped_balanced.csv")
df_balanced.to_csv(train_manifest_path, index=False)

print("Saved:", train_manifest_path)
print("Final training rows:", len(df_balanced))

In [ ]:
import os
import json
import numpy as np
import pandas as pd

def split_subjects(subject_ids, n_train, n_val, n_test, seed=42):
    subject_ids = np.array(sorted(set(subject_ids)))
    assert len(subject_ids) == (n_train + n_val + n_test), (
        f"Counts don't match number of subjects: "
        f"{len(subject_ids)} vs {n_train+n_val+n_test}"
    )

    rng = np.random.default_rng(seed)
    shuffled = subject_ids.copy()
    rng.shuffle(shuffled)

    train_subj = set(shuffled[:n_train])
    val_subj   = set(shuffled[n_train:n_train + n_val])
    test_subj  = set(shuffled[n_train + n_val:])

    return train_subj, val_subj, test_subj


def apply_subject_split(df, subject_col, train_subj, val_subj, test_subj):
    def _which_split(s):
        if s in train_subj: return "train"
        if s in val_subj:   return "val"
        if s in test_subj:  return "test"
        return "UNKNOWN"

    df = df.copy()
    df["split"] = df[subject_col].apply(_which_split)

    unknown = df[df["split"] == "UNKNOWN"][subject_col].unique().tolist()
    if unknown:
        raise ValueError(f"Found subjects not in any split (check subject IDs): {unknown[:10]}")

    return df


def save_splits(df, out_dir, prefix, subject_col="subject_id"):
    os.makedirs(out_dir, exist_ok=True)

    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df   = df[df["split"] == "val"].reset_index(drop=True)
    test_df  = df[df["split"] == "test"].reset_index(drop=True)

    train_df.to_csv(os.path.join(out_dir, f"{prefix}_train.csv"), index=False)
    val_df.to_csv(os.path.join(out_dir, f"{prefix}_val.csv"), index=False)
    test_df.to_csv(os.path.join(out_dir, f"{prefix}_test.csv"), index=False)

    # Save subject lists too (useful for reproducibility)
    split_subjects_dict = {
        "train_subjects": sorted(train_df[subject_col].unique().tolist()),
        "val_subjects":   sorted(val_df[subject_col].unique().tolist()),
        "test_subjects":  sorted(test_df[subject_col].unique().tolist()),
    }
    with open(os.path.join(out_dir, f"{prefix}_subjects.json"), "w") as f:
        json.dump(split_subjects_dict, f, indent=2)

    print(f"[OK] Saved to: {out_dir}")
    print(f"Train rows={len(train_df)}, Val rows={len(val_df)}, Test rows={len(test_df)}")
    print(f"Train subjects={len(split_subjects_dict['train_subjects'])}, "
          f"Val subjects={len(split_subjects_dict['val_subjects'])}, "
          f"Test subjects={len(split_subjects_dict['test_subjects'])}")


# =========================
# YawDD split (30/6/6 = 42)
# =========================
yawdd_manifest_path = "PATH/TO/yawdd_manifest.csv"  # <-- change
yawdd_df = pd.read_csv(yawdd_manifest_path)

# Required columns check
assert "subject_id" in yawdd_df.columns, "YawDD manifest must include 'subject_id' column."

yaw_subjects = yawdd_df["subject_id"].unique().tolist()
train_s, val_s, test_s = split_subjects(yaw_subjects, n_train=30, n_val=6, n_test=6, seed=42)
yawdd_df = apply_subject_split(yawdd_df, "subject_id", train_s, val_s, test_s)

save_splits(yawdd_df, out_dir="splits", prefix="yawdd", subject_col="subject_id")


# ============================
# UTA-RLDD split (36/8/16 = 60)
# ============================
uta_manifest_path = "PATH/TO/uta_manifest.csv"  # <-- change
uta_df = pd.read_csv(uta_manifest_path)

assert "subject_id" in uta_df.columns, "UTA manifest must include 'subject_id' column."

uta_subjects = uta_df["subject_id"].unique().tolist()
train_s, val_s, test_s = split_subjects(uta_subjects, n_train=36, n_val=8, n_test=16, seed=42)
uta_df = apply_subject_split(uta_df, "subject_id", train_s, val_s, test_s)

save_splits(uta_df, out_dir="splits", prefix="uta_rldd", subject_col="subject_id")

In [ ]:
import os
import json
import numpy as np
import pandas as pd

# -----------------------
# Config (edit if needed)
# -----------------------
IN_DIR = "."          # folder where the manifests live
OUT_DIR = "splits"    # output folder
SEED = 42

YAWDD_MANIFEST = os.path.join(IN_DIR, "manifest_yawdd_labeled.csv")
UTA_MANIFEST   = os.path.join(IN_DIR, "manifest_uta_train_capped_balanced.csv")
DROZY_MANIFEST = os.path.join(IN_DIR, "manifest_drozy_kss_labeled.csv")

# If your subject column is not literally "subject_id",
# the script will try a few common alternatives automatically.
SUBJECT_COL_CANDIDATES = ["subject_id", "subject", "subj_id", "participant_id", "person_id", "pid", "id"]

os.makedirs(OUT_DIR, exist_ok=True)


def find_subject_col(df: pd.DataFrame) -> str:
    cols = set(df.columns)
    for c in SUBJECT_COL_CANDIDATES:
        if c in cols:
            return c
    raise ValueError(
        f"Could not find a subject column. Expected one of: {SUBJECT_COL_CANDIDATES}\n"
        f"Available columns: {list(df.columns)}"
    )


def split_subjects(subject_ids, n_train, n_val, n_test, seed=42):
    subject_ids = np.array(sorted(set(subject_ids)))
    expected = n_train + n_val + n_test
    if len(subject_ids) != expected:
        raise ValueError(
            f"Subject count mismatch: found {len(subject_ids)} unique subjects, "
            f"but requested {expected} (train={n_train}, val={n_val}, test={n_test})."
        )

    rng = np.random.default_rng(seed)
    shuffled = subject_ids.copy()
    rng.shuffle(shuffled)

    train_subj = set(shuffled[:n_train])
    val_subj   = set(shuffled[n_train:n_train + n_val])
    test_subj  = set(shuffled[n_train + n_val:])

    return train_subj, val_subj, test_subj


def apply_subject_split(df, subject_col, train_subj, val_subj, test_subj):
    df = df.copy()

    def which_split(s):
        if s in train_subj: return "train"
        if s in val_subj:   return "val"
        if s in test_subj:  return "test"
        return "UNKNOWN"

    df["split"] = df[subject_col].apply(which_split)

    unknown_subjects = df.loc[df["split"] == "UNKNOWN", subject_col].unique().tolist()
    if unknown_subjects:
        raise ValueError(
            f"Found subjects not assigned to any split (first 20 shown): {unknown_subjects[:20]}"
        )

    return df


def save_split_csvs(df, prefix, subject_col):
    # Save CSVs
    for sp in ["train", "val", "test"]:
        out_path = os.path.join(OUT_DIR, f"{prefix}_{sp}.csv")
        df[df["split"] == sp].reset_index(drop=True).to_csv(out_path, index=False)

    # Save subject lists for reproducibility
    subj_json = {
        "train_subjects": sorted(df[df["split"] == "train"][subject_col].unique().tolist()),
        "val_subjects":   sorted(df[df["split"] == "val"][subject_col].unique().tolist()),
        "test_subjects":  sorted(df[df["split"] == "test"][subject_col].unique().tolist()),
        "seed": SEED
    }
    with open(os.path.join(OUT_DIR, f"{prefix}_subjects.json"), "w") as f:
        json.dump(subj_json, f, indent=2)

    # Print summary
    def _summary(sp):
        sdf = df[df["split"] == sp]
        return len(sdf), len(sdf[subject_col].unique())

    tr_rows, tr_subj = _summary("train")
    va_rows, va_subj = _summary("val")
    te_rows, te_subj = _summary("test")

    print(f"\n[{prefix.upper()}] Saved splits to '{OUT_DIR}/'")
    print(f"  Train: rows={tr_rows}, subjects={tr_subj}")
    print(f"  Val:   rows={va_rows}, subjects={va_subj}")
    print(f"  Test:  rows={te_rows}, subjects={te_subj}")


# -----------------------
# 1) YAWDD
# -----------------------
yawdd_df = pd.read_csv(YAWDD_MANIFEST)
yawdd_subject_col = find_subject_col(yawdd_df)

yawdd_subjects = yawdd_df[yawdd_subject_col].unique().tolist()
train_s, val_s, test_s = split_subjects(yawdd_subjects, n_train=30, n_val=6, n_test=6, seed=SEED)

yawdd_df = apply_subject_split(yawdd_df, yawdd_subject_col, train_s, val_s, test_s)
save_split_csvs(yawdd_df, prefix="yawdd", subject_col=yawdd_subject_col)


# -----------------------
# 2) UTA-RLDD
# -----------------------
uta_df = pd.read_csv(UTA_MANIFEST)
uta_subject_col = find_subject_col(uta_df)

uta_subjects = uta_df[uta_subject_col].unique().tolist()
train_s, val_s, test_s = split_subjects(uta_subjects, n_train=36, n_val=8, n_test=16, seed=SEED)

uta_df = apply_subject_split(uta_df, uta_subject_col, train_s, val_s, test_s)
save_split_csvs(uta_df, prefix="uta", subject_col=uta_subject_col)


# -----------------------
# 3) DROZY (all test)
# -----------------------
drozy_df = pd.read_csv(DROZY_MANIFEST)

# Try to find subject col if present (not strictly needed for "all test")
try:
    drozy_subject_col = find_subject_col(drozy_df)
except ValueError:
    drozy_subject_col = None

drozy_df = drozy_df.copy()
drozy_df["split"] = "test"

drozy_out = os.path.join(OUT_DIR, "drozy_test.csv")
drozy_df.reset_index(drop=True).to_csv(drozy_out, index=False)

print(f"\n[DROZY] Saved ALL rows to test -> {drozy_out}")
if drozy_subject_col:
    print(f"  rows={len(drozy_df)}, subjects={drozy_df[drozy_subject_col].nunique()}")
else:
    print(f"  rows={len(drozy_df)} (no subject column detected, which is fine)")

In [ ]:
import os
import pandas as pd
import numpy as np
import json

BASE_DIR = "/content/drive/MyDrive/ES327_Drowsiness"   # <- your folder
OUT_DIR  = os.path.join(BASE_DIR, "splits")
SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)

YAWDD_MANIFEST = os.path.join(BASE_DIR, "manifest_yawdd_labeled.csv")
UTA_MANIFEST   = os.path.join(BASE_DIR, "manifest_uta_train_capped_balanced.csv")
DROZY_MANIFEST = os.path.join(BASE_DIR, "manifest_drozy_kss_labeled.csv")

print("Checking files exist:")
for p in [YAWDD_MANIFEST, UTA_MANIFEST, DROZY_MANIFEST]:
    print(p, "->", os.path.exists(p))

In [ ]:
import os
print(os.listdir("/content/drive"))
print(os.listdir("/content/drive/MyDrive")[:30])

In [ ]:
import os

base1 = "/content/drive/MyDrive/ES327_Drowsiness"
base2 = "/content/drive/My Drive/ES327_Drowsiness"

for base in [base1, base2]:
    print("\nChecking:", base)
    if os.path.exists(base):
        print("✅ exists")
        print("Top-level files/folders:", os.listdir(base)[:60])
    else:
        print("❌ does not exist")

In [ ]:
import os
import json
import numpy as np
import pandas as pd

# -----------------------
# Paths
# -----------------------
BASE_DIR = "/content/drive/MyDrive/ES327_Drowsiness"
MANIFEST_DIR = os.path.join(BASE_DIR, "manifests")
OUT_DIR = os.path.join(BASE_DIR, "splits")
SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)

YAWDD_MANIFEST = os.path.join(MANIFEST_DIR, "manifest_yawdd_labeled.csv")
UTA_MANIFEST   = os.path.join(MANIFEST_DIR, "manifest_uta_train_capped_balanced.csv")
DROZY_MANIFEST = os.path.join(MANIFEST_DIR, "manifest_drozy_kss_labeled.csv")

# Quick existence check
for p in [YAWDD_MANIFEST, UTA_MANIFEST, DROZY_MANIFEST]:
    assert os.path.exists(p), f"Missing file: {p}"
print("✅ All manifest files found.")


# -----------------------
# Helpers
# -----------------------
SUBJECT_COL_CANDIDATES = ["subject_id", "subject", "subj_id", "participant_id", "person_id", "pid", "id"]

def find_subject_col(df: pd.DataFrame) -> str:
    for c in SUBJECT_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No subject column found. Columns: {list(df.columns)}")

def split_subjects(subject_ids, n_train, n_val, n_test, seed=42):
    subject_ids = np.array(sorted(set(subject_ids)))
    expected = n_train + n_val + n_test
    if len(subject_ids) != expected:
        raise ValueError(
            f"Subject count mismatch: found {len(subject_ids)} unique subjects, expected {expected} "
            f"(train={n_train}, val={n_val}, test={n_test})."
        )
    rng = np.random.default_rng(seed)
    rng.shuffle(subject_ids)

    train_s = set(subject_ids[:n_train])
    val_s   = set(subject_ids[n_train:n_train + n_val])
    test_s  = set(subject_ids[n_train + n_val:])
    return train_s, val_s, test_s

def apply_subject_split(df, subject_col, train_s, val_s, test_s):
    df = df.copy()
    def which(s):
        if s in train_s: return "train"
        if s in val_s:   return "val"
        if s in test_s:  return "test"
        return "UNKNOWN"

    df["split"] = df[subject_col].apply(which)

    unknown = df.loc[df["split"] == "UNKNOWN", subject_col].unique().tolist()
    if unknown:
        raise ValueError(f"Unassigned subjects found (first 20): {unknown[:20]}")
    return df

def save_splits(df, prefix, subject_col):
    # Save CSVs
    for sp in ["train", "val", "test"]:
        out_path = os.path.join(OUT_DIR, f"{prefix}_{sp}.csv")
        df[df["split"] == sp].reset_index(drop=True).to_csv(out_path, index=False)

    # Save subject lists for reproducibility
    subj_json = {
        "train_subjects": sorted(df[df["split"] == "train"][subject_col].unique().tolist()),
        "val_subjects":   sorted(df[df["split"] == "val"][subject_col].unique().tolist()),
        "test_subjects":  sorted(df[df["split"] == "test"][subject_col].unique().tolist()),
        "seed": SEED
    }
    with open(os.path.join(OUT_DIR, f"{prefix}_subjects.json"), "w") as f:
        json.dump(subj_json, f, indent=2)

    # Print summary
    print(f"\n[{prefix.upper()}]")
    print("  subjects train/val/test:",
          len(subj_json["train_subjects"]), "/", len(subj_json["val_subjects"]), "/", len(subj_json["test_subjects"]))
    print("  rows     train/val/test:",
          (df["split"] == "train").sum(), "/", (df["split"] == "val").sum(), "/", (df["split"] == "test").sum())


# -----------------------
# 1) YawDD (30/6/6)
# -----------------------
yawdd_df = pd.read_csv(YAWDD_MANIFEST)
y_col = find_subject_col(yawdd_df)
print(f"YawDD subject column: {y_col} | unique subjects: {yawdd_df[y_col].nunique()}")

train_s, val_s, test_s = split_subjects(yawdd_df[y_col].unique(), 30, 6, 6, seed=SEED)
yawdd_df = apply_subject_split(yawdd_df, y_col, train_s, val_s, test_s)
save_splits(yawdd_df, "yawdd", y_col)


# -----------------------
# 2) UTA-RLDD (36/8/16)
# -----------------------
uta_df = pd.read_csv(UTA_MANIFEST)
u_col = find_subject_col(uta_df)
print(f"UTA subject column: {u_col} | unique subjects: {uta_df[u_col].nunique()}")

train_s, val_s, test_s = split_subjects(uta_df[u_col].unique(), 36, 8, 16, seed=SEED)
uta_df = apply_subject_split(uta_df, u_col, train_s, val_s, test_s)
save_splits(uta_df, "uta", u_col)


# -----------------------
# 3) DROZY (all test)
# -----------------------
drozy_df = pd.read_csv(DROZY_MANIFEST)
drozy_df = drozy_df.copy()
drozy_df["split"] = "test"

drozy_out = os.path.join(OUT_DIR, "drozy_test.csv")
drozy_df.reset_index(drop=True).to_csv(drozy_out, index=False)

print(f"\n[DROZY] all rows saved to test -> {drozy_out} | rows={len(drozy_df)}")
print(f"\n✅ Done. Splits written to: {OUT_DIR}")

In [ ]:
import os

print("Drive root:", os.listdir("/content/drive"))
print("\nMyDrive contents:")
print(os.listdir("/content/drive/MyDrive"))

print("\nInside ES327_Drowsiness:")
print(os.listdir("/content/drive/MyDrive/ES327_Drowsiness"))

In [ ]:
import os
import json
import numpy as np
import pandas as pd

# -----------------------
# Paths (FIXED: Manifests)
# -----------------------
BASE_DIR = "/content/drive/MyDrive/ES327_Drowsiness"
MANIFEST_DIR = os.path.join(BASE_DIR, "Manifests")   # <-- capital M
OUT_DIR = os.path.join(BASE_DIR, "splits")
SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)

YAWDD_MANIFEST = os.path.join(MANIFEST_DIR, "manifest_yawdd_labeled.csv")
UTA_MANIFEST   = os.path.join(MANIFEST_DIR, "manifest_uta_train_capped_balanced.csv")
DROZY_MANIFEST = os.path.join(MANIFEST_DIR, "manifest_drozy_kss_labeled.csv")

print("Checking files exist:")
for p in [YAWDD_MANIFEST, UTA_MANIFEST, DROZY_MANIFEST]:
    print(p, "->", os.path.exists(p))

assert os.path.exists(YAWDD_MANIFEST), f"Missing: {YAWDD_MANIFEST}"
assert os.path.exists(UTA_MANIFEST),   f"Missing: {UTA_MANIFEST}"
assert os.path.exists(DROZY_MANIFEST), f"Missing: {DROZY_MANIFEST}"
print("✅ All manifest files found.")


# -----------------------
# Helpers
# -----------------------
SUBJECT_COL_CANDIDATES = ["subject_id", "subject", "subj_id", "participant_id", "person_id", "pid", "id"]

def find_subject_col(df: pd.DataFrame) -> str:
    for c in SUBJECT_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No subject column found. Columns: {list(df.columns)}")

def split_subjects(subject_ids, n_train, n_val, n_test, seed=42):
    subject_ids = np.array(sorted(set(subject_ids)))
    expected = n_train + n_val + n_test
    if len(subject_ids) != expected:
        raise ValueError(
            f"Subject count mismatch: found {len(subject_ids)} unique subjects, expected {expected} "
            f"(train={n_train}, val={n_val}, test={n_test})."
        )
    rng = np.random.default_rng(seed)
    rng.shuffle(subject_ids)

    train_s = set(subject_ids[:n_train])
    val_s   = set(subject_ids[n_train:n_train + n_val])
    test_s  = set(subject_ids[n_train + n_val:])
    return train_s, val_s, test_s

def apply_subject_split(df, subject_col, train_s, val_s, test_s):
    df = df.copy()
    def which(s):
        if s in train_s: return "train"
        if s in val_s:   return "val"
        if s in test_s:  return "test"
        return "UNKNOWN"

    df["split"] = df[subject_col].apply(which)

    unknown = df.loc[df["split"] == "UNKNOWN", subject_col].unique().tolist()
    if unknown:
        raise ValueError(f"Unassigned subjects found (first 20): {unknown[:20]}")
    return df

def save_splits(df, prefix, subject_col):
    for sp in ["train", "val", "test"]:
        out_path = os.path.join(OUT_DIR, f"{prefix}_{sp}.csv")
        df[df["split"] == sp].reset_index(drop=True).to_csv(out_path, index=False)

    subj_json = {
        "train_subjects": sorted(df[df["split"] == "train"][subject_col].unique().tolist()),
        "val_subjects":   sorted(df[df["split"] == "val"][subject_col].unique().tolist()),
        "test_subjects":  sorted(df[df["split"] == "test"][subject_col].unique().tolist()),
        "seed": SEED
    }
    with open(os.path.join(OUT_DIR, f"{prefix}_subjects.json"), "w") as f:
        json.dump(subj_json, f, indent=2)

    print(f"\n[{prefix.upper()}]")
    print("  subjects train/val/test:",
          len(subj_json["train_subjects"]), "/", len(subj_json["val_subjects"]), "/", len(subj_json["test_subjects"]))
    print("  rows     train/val/test:",
          (df["split"] == "train").sum(), "/", (df["split"] == "val").sum(), "/", (df["split"] == "test").sum())


# -----------------------
# 1) YawDD (30/6/6)
# -----------------------
yawdd_df = pd.read_csv(YAWDD_MANIFEST)
y_col = find_subject_col(yawdd_df)
print(f"YawDD subject column: {y_col} | unique subjects: {yawdd_df[y_col].nunique()}")

train_s, val_s, test_s = split_subjects(yawdd_df[y_col].unique(), 30, 6, 6, seed=SEED)
yawdd_df = apply_subject_split(yawdd_df, y_col, train_s, val_s, test_s)
save_splits(yawdd_df, "yawdd", y_col)


# -----------------------
# 2) UTA-RLDD (36/8/16)
# -----------------------
uta_df = pd.read_csv(UTA_MANIFEST)
u_col = find_subject_col(uta_df)
print(f"UTA subject column: {u_col} | unique subjects: {uta_df[u_col].nunique()}")

train_s, val_s, test_s = split_subjects(uta_df[u_col].unique(), 36, 8, 16, seed=SEED)
uta_df = apply_subject_split(uta_df, u_col, train_s, val_s, test_s)
save_splits(uta_df, "uta", u_col)


# -----------------------
# 3) DROZY (all test)
# -----------------------
drozy_df = pd.read_csv(DROZY_MANIFEST)
drozy_df = drozy_df.copy()
drozy_df["split"] = "test"

drozy_out = os.path.join(OUT_DIR, "drozy_test.csv")
drozy_df.reset_index(drop=True).to_csv(drozy_out, index=False)

print(f"\n[DROZY] all rows saved to test -> {drozy_out} | rows={len(drozy_df)}")
print(f"\n✅ Done. Splits written to: {OUT_DIR}")

In [ ]:
import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/ES327_Drowsiness"
SPLIT_DIR = os.path.join(BASE_DIR, "splits")

files = [
    "yawdd_train.csv","yawdd_val.csv","yawdd_test.csv",
    "uta_train.csv","uta_val.csv","uta_test.csv",
    "drozy_test.csv"
]

print("Checking split files exist:")
for f in files:
    p = os.path.join(SPLIT_DIR, f)
    print(f"  {f}: {os.path.exists(p)}")

def get_subject_col(df):
    for c in ["subject_id","subject","subj_id","participant_id","person_id","pid","id"]:
        if c in df.columns:
            return c
    return None

def check_dataset(prefix, expected_subjects=(None,None,None)):
    tr = pd.read_csv(os.path.join(SPLIT_DIR, f"{prefix}_train.csv"))
    va = pd.read_csv(os.path.join(SPLIT_DIR, f"{prefix}_val.csv"))
    te = pd.read_csv(os.path.join(SPLIT_DIR, f"{prefix}_test.csv"))

    col = get_subject_col(tr) or get_subject_col(va) or get_subject_col(te)
    if col is None:
        raise ValueError(f"{prefix}: couldn't find subject column in split CSVs")

    tr_s = set(tr[col].unique())
    va_s = set(va[col].unique())
    te_s = set(te[col].unique())

    overlap_tv = tr_s & va_s
    overlap_tt = tr_s & te_s
    overlap_vt = va_s & te_s

    print(f"\n[{prefix.upper()}] subject_col={col}")
    print("  subjects train/val/test:", len(tr_s), "/", len(va_s), "/", len(te_s))
    print("  rows     train/val/test:", len(tr), "/", len(va), "/", len(te))
    print("  overlaps (should be 0):",
          len(overlap_tv), len(overlap_tt), len(overlap_vt))

    if expected_subjects != (None,None,None):
        exp_tr, exp_va, exp_te = expected_subjects
        assert len(tr_s) == exp_tr, f"{prefix}: train subjects {len(tr_s)} != {exp_tr}"
        assert len(va_s) == exp_va, f"{prefix}: val subjects {len(va_s)} != {exp_va}"
        assert len(te_s) == exp_te, f"{prefix}: test subjects {len(te_s)} != {exp_te}"

    assert len(overlap_tv) == 0 and len(overlap_tt) == 0 and len(overlap_vt) == 0, \
        f"{prefix}: subject leakage detected!"

# Expected counts from your plan
check_dataset("yawdd", expected_subjects=(30,6,6))
check_dataset("uta", expected_subjects=(36,8,16))

# DROZY should be all test
drozy = pd.read_csv(os.path.join(SPLIT_DIR, "drozy_test.csv"))
print(f"\n[DROZY] rows={len(drozy)} | split col exists={('split' in drozy.columns)}")
if "split" in drozy.columns:
    print("  split values:", drozy["split"].value_counts().to_dict())
    assert set(drozy["split"].unique()) == {"test"}, "DROZY has non-test rows!"
print("\n✅ All checks passed.")

In [ ]:
import pandas as pd, os

SPLIT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/splits"

def label_check(path):
    df = pd.read_csv(path)
    if "label" in df.columns:
        print(os.path.basename(path), df["label"].value_counts(dropna=False).to_dict())
    else:
        print(os.path.basename(path), "-> no 'label' column found")

for f in ["yawdd_train.csv","yawdd_val.csv","yawdd_test.csv",
          "uta_train.csv","uta_val.csv","uta_test.csv",
          "drozy_test.csv"]:
    label_check(os.path.join(SPLIT_DIR, f))

In [ ]:
import os
import pandas as pd

SPLIT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/splits"
drozy_path = os.path.join(SPLIT_DIR, "drozy_test.csv")

df = pd.read_csv(drozy_path)

# 1) If you already have a 'kss' column, label from it:
if "kss" in df.columns:
    df["label"] = (df["kss"] > 5).astype(int)

# 2) Otherwise, if you have a textual state column, map it:
elif "state" in df.columns:
    df["label"] = df["state"].astype(str).str.lower().apply(lambda s: 0 if "alert" in s else 1)

else:
    raise ValueError("DROZY has no 'kss' or 'state' column. Print df.columns and I'll map it.")

df.to_csv(drozy_path, index=False)
print("✅ Wrote label column to drozy_test.csv")
print(df["label"].value_counts().to_dict())